# Submit the Customer H2O Scoring Pipeline

Bind the customer's immutable model, input data, environment, compute, datastore, correlation ID, and reject policy to the static YAML pipeline, then submit and inspect outputs.

**Source:** Adapted from this repository's H2O scoring pipeline and notebook submission patterns.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import MLClient, load_job
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

def workshop_path(name: str) -> Path:
    value = Path(os.environ[name])
    return value if value.is_absolute() else WORKSHOP_ROOT / value

INPUT_PATH = workshop_path("H2O_CUSTOMER_INPUT_PATH")
MODEL_NAME = os.environ["H2O_MODEL_NAME"]
MODEL_VERSION = os.environ["H2O_MODEL_VERSION"]
ENVIRONMENT_NAME = os.environ["H2O_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["H2O_ENVIRONMENT_VERSION"]
DATA_NAME = os.environ["H2O_INPUT_DATA_NAME"]
DATA_VERSION = os.environ["H2O_INPUT_DATA_VERSION"]
COMPUTE_NAME = os.environ["AZUREML_COMPUTE_NAME"]
OUTPUT_DATASTORE = os.getenv("AZUREML_OUTPUT_DATASTORE", "workspaceblobstore")
CORRELATION_ID = os.getenv("H2O_CORRELATION_ID", "workshop-manual-run")
ID_COLUMN = os.getenv("H2O_ID_COLUMN", "__generated__")
FAIL_ON_REJECTS = os.getenv("H2O_FAIL_ON_REJECTS", "false").lower() in {"1", "true", "yes"}
RUN = os.getenv("RUN_H2O_SCORING_PIPELINE", "false").lower() in {"1", "true", "yes"}

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
data_definition = Data(
    name=DATA_NAME,
    version=DATA_VERSION,
    type=AssetTypes.URI_FILE,
    path=str(INPUT_PATH),
    description="Customer H2O scoring input",
    tags={"workshop": "azureml-h2o", "purpose": "offline-scoring"},
)

job = load_job(WORKSHOP_ROOT / os.getenv("H2O_PIPELINE_FILE", "pipelines/h2o-customer-scoring-pipeline.yaml"))
job.inputs["model_dir"] = f"azureml:{MODEL_NAME}:{MODEL_VERSION}"
job.inputs["input_data"] = f"azureml:{DATA_NAME}:{DATA_VERSION}"
job.inputs["correlation_id"] = CORRELATION_ID
job.inputs["id_column"] = ID_COLUMN
job.inputs["fail_on_rejects"] = FAIL_ON_REJECTS
job.inputs["h2o_nthreads"] = int(os.environ["H2O_NTHREADS"])
job.inputs["h2o_max_mem_size"] = os.environ["H2O_MAX_MEM_SIZE"]
job.jobs["score"].component.environment = f"azureml:{ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}"
job.settings.default_compute = f"azureml:{COMPUTE_NAME}"
job.settings.default_datastore = f"azureml:{OUTPUT_DATASTORE}"
job.experiment_name = os.environ["H2O_EXPERIMENT_NAME"]
job.display_name = f"Customer H2O scoring - {CORRELATION_ID}"
job.tags = {"workshop": "azureml-h2o", "correlation_id": CORRELATION_ID, "model": f"{MODEL_NAME}:{MODEL_VERSION}"}

if RUN:
    ml_client.models.get(MODEL_NAME, MODEL_VERSION)
    ml_client.environments.get(ENVIRONMENT_NAME, ENVIRONMENT_VERSION)
    registered_data = ml_client.data.create_or_update(data_definition)
    print(f"Input data: {registered_data.name}:{registered_data.version}")
    submitted_job = ml_client.jobs.create_or_update(job)
    print(f"Submitted: {submitted_job.name}")
    ml_client.jobs.stream(submitted_job.name)
    final_job = ml_client.jobs.get(submitted_job.name)
    if final_job.status != "Completed":
        raise RuntimeError(f"Pipeline ended with status {final_job.status}")
    print({name: output.path for name, output in final_job.outputs.items()})
else:
    print(f"Prepared scoring pipeline for {MODEL_NAME}:{MODEL_VERSION}")
    print("Submission disabled. Set RUN_H2O_SCORING_PIPELINE=true in workshop/.env.")

## Expected Result

The versioned customer input is registered, the static command pipeline completes, and Airflow-ready scored and monitoring output URIs are returned.

Next: review `workshop/docs/AIRFLOW.md` and `workshop/docs/NEXT_STEPS.md`.